In [1]:
import io
import os
import re
import json

from typing import Optional
from graphviz import Digraph
import matplotlib.pyplot as plt

from Node import build_tree

In [2]:
def save_json_to_file(json_data, file_path):
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(json_data)

def read_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data


def save_criteria_to_json(text: str, output_file: str, type: str) -> str:
    """ Process the LLM output-text and save it as a JSON file. """
    sentences = text.strip().split('\n')
    sentences = [re.sub(r'\s+', ' ', sentence.strip()) for sentence in sentences if sentence.strip()]
    data = {f"{type}{i+1}": sentence for i, sentence in enumerate(sentences)}
    json_output = json.dumps(data, indent=2, ensure_ascii=False)
    with open(output_file, "w", encoding="utf-8") as file:
        file.write(json_output)
    return json_output

def read_files_and_save_to_json(input_directory: str, output_directory: str) -> None:
    """ Iterate over all input files and use save_criteria_to_json() to process them."""
    os.makedirs(output_directory, exist_ok=True)
    for file_name in os.listdir(input_directory):
        file_path = os.path.join(input_directory, file_name)
        if os.path.isfile(file_path):
            with open(file_path, 'r', encoding='utf-8') as file:
                file_content = file.read()
            if file_name.endswith("_exc.txt"):
                output_file_path = os.path.join(output_directory, file_name.replace("_exc.txt", "_exc.json"))
                save_criteria_to_json(file_content, output_file_path, type="EC")
            elif file_name.endswith("_inc.txt"):
                output_file_path = os.path.join(output_directory, file_name.replace("_inc.txt", "_inc.json"))
                save_criteria_to_json(file_content, output_file_path, type="IC")

def process_json_files(input_dir: str, output_dir: str) -> None:
    """ Process JSON files to an Abstract Syntax Tree and save them as JSON files."""
    for file_name in os.listdir(input_dir):
        if file_name.endswith(".json"):
            file_path = os.path.join(output_dir, file_name)
            file = file_name.split(".")[0]
            data = read_json(file_path)
            tree = build_tree(data)
            tree_json = tree.to_dict()
            ausgabe = json.dumps(tree_json, indent=2)
            save_json_to_file(ausgabe, f'{output_dir}/{file}.json')

In [3]:
# Insert the input and output folder names
input_folder = "LLL_OUTPUT"
output_folder = "JSON_OUTPUT"
ast_output = "AST_OUTPUT"

read_files_and_save_to_json(input_folder, output_folder)
process_json_files(output_folder, ast_output)

FileNotFoundError: [WinError 3] Das System kann den angegebenen Pfad nicht finden: 'LLL_OUTPUT'

In [ ]:
def parse_logic_to_tree(logic: dict, graph: Digraph, parent: Optional[str]=None, node_id: int=0) -> int:
    """Parse logic to graph structure."""
    if 'raw_text' in logic:
        node_label = logic['raw_text']
        graph.node(str(node_id), label=node_label, shape='box', style='filled', fillcolor='lightyellow')
        if parent is not None:
            graph.edge(parent, str(node_id))
        return node_id
    for key in logic:
        if key in ('AND', 'OR', 'NOT'):
            operator = key
            node_label = operator
            current_node_id = node_id
            color = 'lightblue' if operator == 'AND' else 'lightgreen' if operator == 'OR' else 'lightcoral'
            graph.node(str(current_node_id), label=node_label, color=color, fontcolor='black', style='filled', fillcolor=color)
            if parent is not None:
                graph.edge(parent, str(current_node_id))
            operands = logic[key]
            node_id += 1
            if 'left' in operands:
                left_id = parse_logic_to_tree(operands['left'], graph, str(current_node_id), node_id)
                node_id = left_id + 1
            if 'right' in operands:
                right_id = parse_logic_to_tree(operands['right'], graph, str(current_node_id), node_id)
                node_id = right_id + 1
    return node_id

def plot_and_save_graph(logic: dict, graph_title: str, output_filename: str) -> None:
    """Plot and save the logic graph."""
    graph = Digraph(format='png')
    parse_logic_to_tree(logic, graph)
    png_data = graph.pipe(format='png')
    image = plt.imread(io.BytesIO(png_data), format='png')
    plt.figure(figsize=(20, 15))
    plt.imshow(image)
    plt.axis('off')
    plt.title(graph_title)
    plt.savefig(output_filename)
    plt.show()

In [ ]:
test_file = read_json("example/example_ast_structure.json")
plot_and_save_graph(test_file, '', 'test.png')